# ACE-Step: uji konteks dan penyambungan bass

Gunakan **VM Colab CUDA yang sudah punya checkout bridge/runtime/bobot dari notebook 10**. Jika perlu, salin sel-sel ini ke notebook pada VM yang masih aktif. Tool ZIP ini bukan installer model. Helper bridge sudah disertakan di notebook ini. **Upload piano asli saja** saat diminta; tidak perlu mengunduh atau mengunggah ZIP tools.

Semua flag inferensi awalnya **False**. Konteks **90 detik sudah dipilih pendengar**; 120 detik dilaporkan banyak noise. Jalankan sel persiapan lalu bagian chunk seluruh lagu. Probe durasi bersifat opsional dan boleh dilewati. Kualitas sambungan lagu penuh belum divalidasi.


In [ ]:
from pathlib import Path
import base64, gzip, hashlib, importlib.util, json, sys, tempfile, uuid, zipfile
from google.colab import files

for dependency in ("numpy", "scipy", "soundfile"):
    assert importlib.util.find_spec(dependency), f"Kernel notebook memerlukan {dependency}; jangan ubah environment model pinned"

# Generated from repository scripts by scripts/build_context_bridge.py.
# BEGIN EMBEDDED CONTEXT HELPERS
EMBEDDED_HELPERS = {
  "ace_context_chunks.py": {
    "sha256": "1819d968044da24cf159720300ad9bd11f7d008ebd55036378f23447c1cc73cf",
    "gzip_base64": "H4sIAAAAAAAC/6U7247cxnLv+xWUzgPJ3R7ORRfbs0sBPorOiZHEFmwnCDAZEE2yZ4cRh6R52YsVAfmH/GG+JHXpJpsc7sqB9bAaNrurq6vrXsWXL19+//7D4pdWVU5cdkWq0kVSFq16aB31UKk6O6mibRxZpM5JyaarVSqcRp6qXC1knt0WKnWaNmuTY1bcBhcXP6uuUY2jcpW0dZZEhy7Po6aEd9Wj0xXJURa3Kg2cj7ksllVdxspJ1UF2eeu0pVOUTlYcVK2KRAUXPyqVNk7RnapH2DPJ6D9E8pDlSjiHw6lSt8Kp8q5x2qMChDNApbh13v/rP3zv1F3RAvbOoaydxUI9qKRrAejLly8vslNV1q0j69tK1o0yz0fZHPMsNo//CWib3yfZHi8OdXlyKvgFkxz94iO+0L+bY9dmef/UxXC+RDXNhRmiozgSzlT1s8x5cLg58B501qAB6srcbJSUda1y2eLBsyKNKiU/DZDP6U3wgBi5gkv5/tcPTui82qxWq4uff/rpV3hAxL0owq2jyA9q1ZT5nfL8AEiCV75b7y8uLuBu4Fhy8+athwf3txcO/EuzW9W0oaZXoCf49O4+a48MnBYEZaUKz61j1yeM2lrJE0PBf3g5cV4mn+DenaxVtZfLU5zKLU8EtGTqrVeb15f4xxex6/pb3j7oqhTI4dFy3rtWbVcXGr3gqB74F2DGJ7mvYYsI75WQE3cy75Q+Ew6EA97XrULKyvoxxOcAjxU13eGQPXhu0J4ql7fspwUMHCXHwx2CFC678WgLARcGNA03QuZ5eR8Vsgj/JvNG+VfufxRnkGpV5TJRjIhGPZePZdd6bdnKXDhaRMPvVgJYs77NivAt8IVMVRi8+Rbed6oJfywLYBZVqPr2kR70SQmlJvQ0EMEABK5mTLKD48ni0cuAf5pWgix6dyIuy9x34LqKsnVGb7ysaMUhL2Xr9xNQXoKsAU4Fonh3Pl30HV4y7+73HKAXrFc3oUboJny7Wpnx1Q3idcM44iD/ulnjb71gsbk0oyt/4K1aZo1y/g33+1DXZe25qFDMItjxf//7f2CnxpDwXbhumIiIp7fShPGR3LWCt6vG7Qk0oQLfCxDCHw50iWJ3E9IrOhMNPIffL2VXJ8o5dU3rgGI0GDqNAqRTs/tfnB/gBLcKaFGmKnfSrpZtVhYNoJ3kHaCvQFtXMk1RJTs/qjuYqh4SPDyqyVr9BjfQIilkFRDIk3zITt0ppGuDqwRkNJl8QlpLSBrWqK48Jgy98a+JOqGGAFcB0wyR8M3N+uvnNucxVwN4ObmSd2BIctCfgLUsgBqNUzJMR3ZpVmpyJIBSG65xP01qjQsoRdgHHryNoIMlKss9virG018iNF/DkZVMsjYD0djh8BVMQKAZcsNuJWifxXrPYAkNZGp6XaNV82iGvx9Oj4iF660NWePGk8AiwdBqf+1UZZ4lNMG8od8EBn4CqzkowXpvGDGbTOS0fVJO2z8gp+0gloNkrm7aG6LZkswInrnFMyMWBAd/vAgbMEMq9RrVevTmWUH8K7IRqDoHLXTTMzyjIcBPyIBF0dgjTHI+4BwZcDbyL1o3ffeTC1jzJdlbq5MEoMUti+ECKb6DS+wntMDKqg3N+BUzeL/scri7XQbvl0138sZj270/kC0vkfu8HpxmfsG7TxePlh6z8JQVw9LJzhoEM9ki8y816AEAcENevjtm2xnFVzoH8N6yGPyM2NAevb16qhFYApda3+bZKWuN3rHIhYgeM4FHzUvBgyMqCDyN/RYEjrAVeimPXm0mZygr0mThju+h5ddjpuOD3oSjKTfhMdvbpNCQtjZHo8AR7vql+KQeQ3Y4nHYr48ZrF+YwWiYfQxe8uDvgvTQylHN7mCiOkx2AWTP0S5owB2/UY8YkglytBaK6XIKVIsHZzZNxP4vyAHiMtccGHmQXT80PrCZW/lWwWm0uR+dajulNel2fEz0TXh8VStYRr3CnYN1Y5qhR0giFd6AFMm4gK3D3UmRif1BqZpQ38i+ms4mz9fCxKz71WpDEW3hSxD5evgL/WdXo8/2eVSQogqSFRHC4BlB4dSsAMHHgSsgFqnpiPDbTMar2gQbGfoZAynRBq5lM/QzZNEoHAWxEzBK/d1oGMtAJzNE+u+j6PbjbTLhoMiIC726lfoQ57jYWMKvqWvOSD6DHaAb8EaM7s/655AVEBiV3O4Pml5F//NnlAC5CWrpbUlAukSY61BIUsrtlQrn6cBHoosg4IVvjN16MsSCNMczSXqWL/tQwik/CNYIUGQ5xt+YX7EkEhG3o/y/a/8VAICKzPwpEZCtFU4fNgSMF8uplfi8fm2iThr/WYEXS9rFSoUuG79Vm8OCa+kVIFg2YDOFgGFMpCHtehBtj+4pqMI44xw/Agfe+7mPyEuBECGVLiLs+/ePvI5/FBCoAUp+vAqcfAi+vIRdQOODsw+3/MU+fgjjbw9dAGAYHNTxkxXmChnmKNUzreTQ4fUqz2tPxIBOTQuyo/KTDF55dZ4AWMA/Gg6EOBvV+NOEW1QcxYkQ8HTL8pccs7l7x5IDDqwD0EMSB2pxwTB0kZfWIpzT0mYIcbpW3P3v/Ihyh+dwN/qQnOrTS0SkL5/6IUTrfEyYZ0HBCnGbCtwQc1sKczNUnupd3rvbZ38uiLLIEwBIjoEeH6g0hoKOxPGUPCLUs8sfA+TujDx4RpgNoJ7ghVd+pVIMzp4G7Jpw404AQQPyTo+KMCJogBLUAdQRWDHM1lBIAjQhuNaVfGo4AhmxFUHeFt3M5u+IKd1GUTQuhBP68gz8KqUTj+CdzBYTq5rphQNY8wtZ+VmnBpATWbnB9spXwX5Wcomb9Nle8lmjp7wWcI/lEjMc0JsqFlirgiYPHjDLQu99GMoa7PqlWosyFFKCDQkgbj8Sgn+uTHuEo3h+5VmZtADbR0/cbMbe5wF1jth/7BM/4vwaqk5ZwZdojT450edYVU1iIfGu5YnTgMVZpeV/ESoLHJnZ7/2qCskrI6aF3YyhnvjsrV2UMv0dmwToV5VxCuoMdG1masWD/5lt/i8Y2V4VHU3x+e2Xe7i33nPQg6WYPlG3zW03/n+BaObFzebnxNT4oKqHOg1iwZ9IYAk8hGPdhrUkYNcBWJxlBVNyg1d8Ay8m2a8DRQ1lUqSt6VcG3OlYdQt89KV1b0M+ZfcQlhkWYZwUJdBuyPAYff/7wy4dfzyFYeszedqreggLoK84nz+J/tgmwHZzkR517rRWmesFyLXt2Cpz3VmSuswyNtkeNjs84hB+lHn4A3wfdILAf2i+6JoOWFV3WPnJSuWtIKf7WyRzHugLuJTtkKg00r1tJO6Nd6TZxxBX4c2RTcUDb1DN6eIcyh5iHF1lpPz2+JLgkLPN05+12rj0ErufEemLePAF5Cp8DpzXHGKAe3I+yTBWH6BFlf7M7FbWlxnfYl8IJO0/7IjRYPGfp/t6jZXSMMXYATjVw2dqOgEkYey64iclMghqN8Bi2YzbJpg6YXpMUW/qXjnemdBkdTr2yA7KcIb1NJi2IQx57QjILlE0trbBfhLMX8RzxBoP+R2kHmPIuLPsueJoj8Se0eID98Cl2Ix/9WeQ+0g4EkJEzoYCTZgcQ8oY9C8t/ORcWi2IzYjZ5qbWi5gmS4ceIZN4WOkEjmkfAawG7Y/OCkUOXqRChl8JyPmOXwbM5hWZBX/+IcQlq4+ve2PM+LIjscIrPX4ZMt/2aLQHacyOyYA6QivYkfXv9rZjbtFPaY9M7sSdm6Z/QDbPbcPgIwInKu1FQuR9HtGdr+xTQEEgCIG8ECUPR/WIOuP8V6PoMtzIjsME351MmUexwismLEZkn0gIcMT390ziP4GhlwADGHKFO/bU9G/i9xz0ccrMLzI8sgSmwGtksDe5LNL/OKWvIxxsrVNzHhLpdwZLTeHVZsnhFlm615UnrSUsH95NtPfq8JWRR4ELiISnafCjk9RIZDCAC9MtcU9mTV1zZw0GrrodgsJKQfPLwj+CBf/7p/T9FH/79v6ynH/86cDQQ7xaESQe0ZmszyprgbPLImNgARiqDlO/oLYWyjedzVuvzl1FNkuiPOSeWQZ2T2I/9eszDUbhiuAszPXswcaQWDf4HXh195mkvadrL/Xb1Kv3ijhNxtE6H3X2kPYQ/VjQCWxN2+kTbM58ukeDlpoNnoyfuYOF+6rBMYKM15eXnNpTM53bWGuICMgu99aMDu+d7jKwD72Rbh7P5FcTbIIrUWuAY91BDx2SoOORdc2RCXWv/Uo2g/MX5WSUl1sEk5WYaxFW2WHJtHXlo8UVBPmxddxUmwg3B2P0MxinZErNn6OH18abNOwZuVugChqZbcJuXMQgMv44uR/pkiIjIKOpJM5Zw7sa0wsqYs4Gl0bO2xEK/t+VhbO9C295tZ4P20aVp9Oxbux6oot9ex7Dhpynn9tNMTet8O63VWfFXj+0RXH/yJVEdgijDdR5kljuxOmAVLoFtqO0DrhD3fTQ3cB7odKdY1eHq7AXndvQ9gcSaO/rMK1hW/V5lbHn4Klyfy11PheeBPXVocw2s+nW2qwcqyHY9bd/n8y1Dgvv/Zc6F7pmZ0UDnUtxj+JQgjzQQq02zJjgPbXz/2m7WsBW3ME9jf1QPahuaxp7d3DHKM2xWl0UFknG7XnmYvOBRni7WarGmjIO2xadG5xlmAE0TFvTcYRaXlujMM4zS9Lev/QFuhR1TI8insig5o8LAJPBauPZ32+23+0A2CMqzQBn3FRMhuNS/efuaHFVAmJ/hJK+3fcb/+Lu7pYYQF5u6yjRLINp2t6vgS7/7IsS/vLvOtyVh33ZEUIX+k2JCvctzV4ATdyxTeDqA57nrscGyqAaxRArosre3FsMUGJa6PGbGtGufl/fimN0edYnv9Wq1Wm7erLiGY429euMTPJnAhjrRg21RIgqHHil4uwOAWwSIxhnHrsBjurejR4RCbyg2hBWr/U24+sPki7EdipkC1hKgPZ4ZFO11Lm9DtF+Ie8U5XzLdOIkq+LBg/y5EEJfBd29MN0JZHLIU++DCntmSPKvoNPJ2vyQcxUqs/UltB1El6sA0VrgG0rswePOG3Z2ZswzzTM2FSjf3CugGSs+urEtUVXiteVY0FbYqrQQ8VdlywxX4c9a3ccSjlHAvCAUYXCAyewQAOnAyalrQlDxF93Cl5T0wy6EVNSIlctIzoub/EoX2W9xnae8qa3VJLxb0YrncXGNpkN5c0RAfKUS4Oxpe5HNKEYtw82+Y0+OQkNIg6idB1E+DQG7gMhSEXoSb2BBDxmejM0X+v2HnElh32KQBjnwswQmIS/DHdIbU1PHvpbiPw9HlMs2u4xxJIy/v5VV8eR9zNksKBUdDPegL/C/2r2t14FYhUn+ekpdKXqn4UsX+cmP4t64HzmWNNqPGLmEFrY/ZRUchoR39d6C/vhmYlfkHzVgIup3wQGT9JeDisw90sJasAl0n+Qjj4N391mWKqmlam6VOC6qnybgNAIQRovPsd6XpheFbnnO+BF2puMvytNO9UsC6bejRgbHAzvV1wsy/8vDZWy9wH193QNXo6KCOGRp4VpPQT9erBR1JfHaZYa2KKz2zUXaRTyMgQJTGB3gH1FDSKrS4xIWTCbFP9Wa2s1gjxgfh0n6Dh89r3C0dxoJY1QqUBOYGcnUHLhknumgqAV+ipoNb07YTt2pad8u3j799o1G4UziSdS0fG4pVBZm8RsurLv+Pg65r4k5tC/jVqKi8t3pItEXkpRDK93WCZk5iPoAP86hjPdODRNmconXtSigqst9VDTrLm0vEgThOFN6rzdCfxMpJF/0Kh1oXCL3x0Y38kwdgpP0P+mtaS0wr1uxeTEvWz2UvGEvafVmgPaAy9iRjQepSxMYLtdoa9sIeI3yHLAtRcie38V4XjuTsWbbx4mkFiXZg3B1iN39ZN2+dd7AXjLFupzK/99dFecLknFbfo/MMPSVKgt427Y+gpNbBG52fmzQ+7Jev/ElTk9UT5O30bvshG3zeDbEnFMNnW4/sf6RUmAoa/IIRFvrxih/7DqTNyipuYnDcSqCjRD8IdLRzlPmBctemIdO033Zt34VHfbKwQkH87qj01oqR2zo7jcR1SiKij7EUEwrtEpYa6vbqh0mX9hb8nbmrEYtc4b6z9EEdnlyxleuB+Dds8kfShSBGbWScDDFozLfWIVdyuow6GOAKi5ZI4qRZk8g6tQSHPEVkIM/2aVgTMGOa33theTqJdmvmaOPbDWEP24fdq/2OVfB+JLDavGDM35C7aXHAD2B6sT9AaAdC3z5YjESX7twG7rWjJkHKk2TYklfSvELdEzsEU2EfM+NWasnng07eLuzLMLpAzo2e6ZR4O2FzotWw2X4XL+YYZnbZ7Mx5NUb0tG4WsxGjE5r+s5lD6FL5K/9rZ6Ts1NePMj+sS+6vBnB5Jeoq5PCTMfaFDkZxI0skqelnvVmtLsnVg3h549XVDuOL/TLXP8jJMQ8kZ2bKxHXT3njbmCo8pUHDzGgooxxC/czODtt5PAsb21DyELotPBA/0RdHXhIdLMQjk0/Ej3XFBzZuDB+U/j4B6ySLDjCs1V2m7kOPjrF7wnnaOzfO4hUFkXFjpj7tQO39d2+fauwb/UOxpy+xbGcS6Y3b0BvwmDFEnofGJf6PeO4l4aCrlVjEf0S3R9fhA+cDVuIXFfZgDU0ptx1G6q0C9+goG2UV8QNLt5F1Np2PdPQhJrdiARAKDP0Bb919pm0RVodCtq0CAuElLuO8PfxgdloHtuPMyy9xHTjNtLu7pf+Ei7lWbHhQMq3L8sSlpy1NxeugZ8oPuFv8+2R3peMi6dyt+wt/jZVm8hYborKkoV6ta/xmTXZtCUKSJU6Sy+xEnykcyxouJ0m6WiaPAhvd45oI11BPVvJIJbBx90PgThxlbMX6M+WfuRqJN19XmamwstKx3a2vlkUA4fOaw0yNZLYGoc0twJjvd3i+lDLn4OuanF2UGMrxwxENz1qdZaPsJqyx85r+KDboa9Z0ZSp1TSCiuyj7pPHM9z56iZ7KHzA2132zn5O1JsktgTJHEEnexbUuSOiSwdOx1XlPpy4OHvijtb6v5ryKziDIaxRNF3Mr7cf3/xJtXrtPADllD/QVIq8P3lx6wTeXFmV1T8OVbt96BjRyHNZF6QOcMXKu6Pex2e9PdjHqIxzcz7jtFzqB3zcp5ll8ql6Bn6VwSUxj6+82n+bXwlR3poWRzoT+IsYtdHE7rbmsY3C8r11ofLubpgT2kxBjegsHAhp9psUjObGuVfsnOj32ikFOW/f47ZV+u3/uvnSZybTZ6a66Xi6EesD3OmwOrT2sIrvpknuaJf0n28LY/nBfGOMyyrPwkFaw+H2PiYqr0HwJHHxf33b4xfNHfKq9FLRGndHHImEUpWUSRf41HD4E+5WmEbIbTWxwZhuC130CZyHF7X/rMrD61s1T4xOsoJW8zHO5AYeaomhc6v09d7FgQXEFURrrXxOo86tK7G95asnQCzldZj6lNAzGIMhoC/2Bdvjd6ok9Ob57du3bZ/ZGl+7ZxcGbb5/YGRs9reMOMoYNkaw3qMsZRBUVv9DcONIZ00vBpXDLZ3vRXT1PWmPFAMILw/ngkpyBwuKeDcocFL/NnttaV+JAbqmHF0FjlgICMuWe7cx5AzjzzMZ9Gw1vjl8EgqeSJRh4v12J71YCPP79lNHO4XB8MXdXOrcJkxuQEiIprmy8PkGHT4GWlJC7bl37i70GwZjvIWiyLoPSb+BvoSHwtymacXhwnGngMTtZqJf2XyCrfAYhJp7VD/KMw0Vrhy6aoY1jaP8geNFnmmnozy7L9R/p+OBAaahZP1mrpolnpWoatSvVmsDTejOhR0Xn55uFxLMb6wI13xrltfvi8+j8fW2ZRvXD01eCsjtOk1rLtvpEVt/UcJiZ+6GP9iLBq0T09GXSREOuwSGfzOLuFOuTfw3YfPNPNV9AOYpQOKMIjhNFaH6iCL+SQjN08X/qghc6h0MAAA=="
  },
  "electric_full_song.py": {
    "sha256": "5629f87436e2168a80fe5e34327d41a12ebc3ffc1ac23d914c54ba21d2c55c25",
    "gzip_base64": "H4sIAAAAAAAC/70863Lbxnr/9RRbnB8BFQqk7DpnyoSZUWQpVo8taSw77VTVYEBgSW4EAji4UKZVzfQd+oZ9kn6XXWABgpTipvVkYmLv+91va8dxblSyiOVRFhSFOB7/93/+1w/jcSHOYhmWuQrFL9A+FCoJ4yqCkaIqpAgKEYg4TaSosjgNIhmJ0zQOZmIp40zm3sHB50IWolxKIb+oosR5mUoSGJdXSalWsEQSCZksFKxxcn0xFA+qXKZVKVRZiEgFiySFaaE4fX/xXXFwPB69hjOFQeaJK5hRpFUeShGmSSm/lEOBJ1mlkYxhSBz/KJIUhsCe4bJK7nHzNBewXBku4cM7cBznQK2yNC9FkC+yIC+k+Q7TbGN+/w5rmN+roFya32lxMM/TlcigLVYzoZuvrSFFNcvyNJRFUbds6p94f16h3GQAJt18A3/H8jJYySILwvpEVaUi8/shWNftX1U2V7E8OPh48ulMTMXrV+Px+OD649nN2Sf4fDwQ8McpwqVcBf5a5oVKE2cijofCURH8cKTG8NEMMOxAM0EQe2D3opTZ0fr4DXZKZ8iLRXKtQokjwioKcEqWy1DplZ05kEL5+hW2q6Qo82olkxI7zAaFlFEBDbfjO73iAi4XJKH0C0AcrvxXb0zT5zKX1A7nwClvsDmd4xaRX6Z+mFXQ+imvpF4JaKPU57gogBZLoLU5YBsuXm5ioENDz3gYAfuWQS6CMExXWZAoPKknTkSUb4YiCcoqD2KRxVV4D8uUSF4ObWP9cZBiRRjLII83QEdArRVvG5RlEN4D0xRLRFSSllLksH+APIF0HwAnzWJkHmAWsQzyVZqosPB6NjkRBZGFWFWFAhjR8WPkGpUADyZpvtKNR/pOuVwAx8G68zSO0wekfuTDTAVJ+l3RswXvD/cGusTReMJokwQrPBKg/ung4CCSc+DKIkyBkPw8TUsX6WIoAP+AqVxG00sA0mDCuHCcc4XXFLNcRQvg06UM75G5DZMjSGZpei8WcToL4gL5EyipAiJdS4AmUEOS4FkMU8MKcBaHD49bA4Ujv9ExBl4uAeVr6Q6ofxXk94B2GOI6ICLmalGMAECyLIiaPeRrJMcyTeNiBLBa+XIdxCMQTV62IUINc5WV0CfLKvO1yMI+3gChARNU5KII0LfGPzmMzwEvcexSlxgBMa3ggKrwkVndAWAlpzbEnz4oL6rmDTSFKhBGAoHarE4r6nvXQwee/AIUHIFYzt0uJPS6uFTveenMgQJwfuQ7nuV5mrtz5+PV1SexCQADkcpUrJZiVt0HSYNJxuxEPOKKT86gCwJspjbAZAQ8XkrEB8BTnwzBAAQaAZ0CICIVlh5KxHu5KVxNW4QrEM7AmgPrzDiTIAHzXF5iaJYaAepINTh20w1guPhVJjIPUBMdfUCiKrRQ2/pzyBM9JE3XQTWyCjIfBZ5/ONKrD4adYSAxfa2NfB7jHzqDDqRVm2os0HhBxK0N/hhMuQyiDQDulm5sX72Z3L4GbGJIz1nLJErz0cnp2dENyvNj783Ig8b1aKaSUbYBZkwcizjbK6Ec2LsUEUOWqgQ4pasyRsS5XhHMZSmTIs0Le6M7Jo1lCuqE6ALkpIxcvi1csrlczRuxTFw9YSB+FscW020TsHMSgfCRM8B4FnSJ1hMXhRJI4f67i8tPwM3JAki7COJgCf8vKwNjNdmWlvaf7wXKCu93gIALNOKCyhuaSw1qBBI76FZQfHgXc3GgQkk8riUswcxnvLgoZTUBcQvACdv2ItaBXte50eKL54GAkA5umxYeSZ7pVDigmHl7xyaFA0tg6Mk1zvYDXH9pgT2TcbUSASAhw//99gGh6Yl/BhAnKEYKGEOytTYHwQoQs3QG++4FuROpWnkUo+MxMB2Ynb5R7v68imMfedZT2SaZDUUcxJUAvQyWpliQCCiU57RwwxfVKCDdvbSBD4abhwRxOwHkAScCll2Dh44GQblgd2s14gC98+rzHK07WwqTEYFmnZdmQOE4m3qBsPKZMyBLG6yFdGLLc5eavIUsaUEQa4Cfoahbk3AJ+hPQ22otglX2oCJQHoOB+AdQjmg4DsUr+K9XJfwGoDPoPfuSAYRBMaFlIVMwNu/ffRXXpx+Of9iW/c059H0H+vrA1mjogAz8Kn1wHxYuy0owPcAOlyVCDwypKdh6UYXCOk2mx/CxCFQyHXt/HYLG/OLXXeCnNBYHnDBWoSrRLVkcaVkM3gfb0AUcPU3AsphtwBoC7fqQour9EcTKIgg3IgbTBxwP2H4DpAokUNbWBhnqQASIZMK3sddz+fdK5ZIaaeQaAUZmB12juQTfoH36mtfcINm4YEODxYyWsLsGzQcrDlAOIh+2u1w4Heg2tLQH9RB0ToBRwdxVQAlrNjLWqCX4SG3BTr4QnE/8JMZmhTF8mJOJn6atk+I3gNoeitfB5uMtqWATDfiAgs/EG/5sI/anqb3HT4hM0ji0NKr28fD4TtNW4015IC/cWxvyrjOfrzK5QN5zjtBrBC8V5fLRGv8v8Sj0yY3KYRZlwhv02wDOEfkrNA4PTkuXuqVGIDYGc7Q0nHUK8k5OH/HwT86uRYNcL4GMx9NDPBR5TEfhhD2qcOUXxz/EUo9lzhjcDVmNTdHlYagAkVZg+0+NVNEjQfDg+oa8glnh6pFHNfBJg+qBezA4d97KMEXvvqaFlSqA2sLlhMgfdX4kHk3vExA6UMgj7/fUFrPcqCUBmK9o2dX8D9o5q3r53/Ir2OnOwQdJkB0MHxbL4NWbH4biIQdK88lp/2N8C1CymEzz7i4e1N0v4EMmnRa7PcctQZdfGpVcY8ByDdhEsw7XCJxdx29GtG6wT+fWUmP7ig096e7jMYqFjuR4Vki0kE3CCz3Wh2UaowCJkT7yBjYmRgUWBIj5qNAwSnO1UEkQ+8ugQDeJycIwOoMxmafQgyTiYSChcC3JQuzlMyFuixgYNZMsYzpyBZz8Bx98lFzJnT6FM0c/vZyai9LEFGWHo51RSyYBq6PuYk7XrL4OFNg5Mbq9hDEXr3Krl3Xubp165bsB8DmRj23MtdFWL0d4axb/CfEHLa5NbDWhNeN+Rqx64zd7EXtDRtkWtkQ6ZwPhR0IyGmyEYzgiaGfYCQ4LQhxPslJJBS7AC3igOdr3Ag9mabO9R+QwIqxGkRqJdAd2KJ6rkW7ySyjzrDRURvThre4jhZYaOqcFYWrIkU4/vZ+eB2BY83AMD6HhztPQJjRk6oHRxwZGmK5WeAmwLf9ExbatxO72gXFSH+N7OEePtrtrnZUHkfYzyg/sMzrIN2k5gpPeo6Pt9Zbb+o986Ub98RI20QMJHZKKQznEo7Upc/ic5qsVH5EqBt9KJM4domcvaYL+5b1Z3Xa0MFLrS86hqXCLVms61UdhF9LwGpiivPf3Zu9jsA8sAwHcMwrqTzsTD814ms5o4bCGDkI1BM3tDTn3eDSMGsulMUww7Ay09sDhD3o4nlJGdiAMvz3wIIEDg1Xhuq/Ij2GHBiPFl1eXZ0hniAoM8eZATQijRtE9LFHozOI0vBeTaX0iD2MQmqKIgCdmNzIvuCcPHlya2YqyaVjunOD8+9gBwLoG5ofiH/V5EOYKKYJR8aij9r+1gvYa0/D12Ej7SRtvGLpEXod2+vtphyk6k0FJgXjQNU6UPiSthhAILNJf/fMfghzPSmOcE2SRI/Ku7ItQFkYlYKyhFOXY8uhebka0vFjJMoiCMvCcuyemm9p8cxvishc0cVu7bbB7qjYxIzPtsT8XshOU9c2bQxQE/BPMMVhU/zS0hvhsdtQDW1YIE/ZT39Jwl2aqnrQPDIMnze8ajEQ2tQlUr9QyivCcyIlbG2n+3MK1vuXW+A6bYmAhlj6GIWAMM6FDgsNn8odW+uzZwQK/LSVNNsrfQ+kyiXzNS80+uqHvLjZjNPwkI3uPxjw2d0eXDjpAu+0hU5WgUaAJzaCk5f2YRu3/yC8yBNvG5zAEBYwaJ+iQ8iog3cgextyrSaZNOZXWRD0+SkzHosECg3SIRmdVwUCB73jDqVWhVeoRiK0AMyqULRudX79+NdJBEszt5CqyEi18Ckw5cFaRUxTYZOxCCiFyW+1K1adl3wNc+ToBGEEHHB9tTJ6EUSLXzhZSZKr53qca/4VgZ+9Haprc044Nx5tpFWmH+Pb5lewQVBksL4NVv3+pw5dg/69VniZ0CuNmbnfRPHvotG+UfTIrWeQ67879d59/8U9PTt+RcoPvfz371Hx/uH7/y8np384u3+qv06vL84tf3158dCxLPS08vdktLo0xTWtzbjtoD/SqDIPxrj7A1fn5+4vLs6lzDPt8+nhyeXN+9fHD2ccbu0drthYM3T3xazxygKmVVtDVA+zKvET9bgdW++ZrXWpy7GkeLm1jkBo8TONgSLt2GJ6La59+fnsiqqRxMDBtRzFujTlSc6fXn0cfrm/EPIjjWRDeO62z6NyIB/ZaFAPFYkwXJIv+tEfiXLhaYbfBPjiDufpAexWYwz7ndPs7Xsa1lvROQolw0V3WFVH2+JRDX4Fc81mkIiX7TTLHLWQ8BykU5AsQs4eH9w/4azuXxe3MwboygLiXiwMmveC0a020KNE+TyEQ1E47EoEn8SKskgAqZQzqIoPWsCzHYzu316Ya4U7YG71Vn0boUY9+OzmbNkUK87gqlpY/0YliF1VGGdVnwdUFVBvYAB5/HUifbqFBy9DSoS2z3/b1dmMUoNHGfjsCoskIRhmCwvm/8G+LgkyNj+5xuwt0CcfH8IHvv4hAavCZSTvB1Do6G/SUKgbU85ePoGsNpxAim/77T4F/uOO2rhBBice0Z7VtzdJY6Rxs/y26Z8aN6EuXLWzhAgZ0cEAj/yJOQZCgM6DtBBfcoHWqwDEz9Urf2VVSbHxTkdQR20ec0PC0wkHJ4dnJF9h3KxdjND4ZzS0PzzSyUbDQifS0HddomrvDdLCkLzwClofMlVaF+pxZHCSu2VIrBdsGGmL5UHe60VCYU/g9nRkrEbMKht382hyYmvIlZDDi6amhBy6B2uH2AJm2CpB03KddwzTFLBWsMnUKEkF+RhZb13a0oAjAsy6ijcimRRstQGwAo06NmGs063S/ZtWkN9XAbIQXW5j1bXcDRHABDcjBRM0BivWGewpstJjT0D7oAFFDDyho2gBjWFPgtCGBPqTWq3Whj/VjyNVVDjKXgif51HKkuEfjo8w3jbjQ5Gc4jvm8iwYMw2Ql1kTKM/qJCETtjOrNWmuLMMHrKKti6szBjgBOGvKM6dx5JL1AXygs0fzy/SdwH6mpVVmD+pTNQpRIsXX2byQs44ugneFTpGsrIRumFbIcZWCP7TzMizIr/x+JuzoJF4AYXU3RK2SHdPpIh38adrJz+2OU1rp/OCOn5t1cHJhDdIh9fswVU2cEzi/VXSDn5Wupq2eDsKyVEK/OKzoGfSAH7oOF1N7oH0YRuhhtpJOz4dbloX6M5YQJhTqGTdWoD+dcK/ngrbLXiELXWakvVH9hRuJ3a5Clm/8PCKNm8r2p3RZmYzWDg8UAU1ppxq3H//TqvrvmbpyDlAK4omzmX1QtU7ht/01XYWjI1kUqoJKKKi69rypzEII1bNFf3gJ4Gw8taA/23Nbeby5lhLpge0cbUb24s5BH8QRdfuz9m8rOsT6oBhXf1/lCkTqOvGJqrx5/ce2/PTt/D4z3lstbwOxVa9m22+ZUEBqDHFuz83uo4YxmJ4IS7GY9keOszf5m2rD+ZTiljaCJHZ5xbW1dF4gaFTRqxXcsU2e0JV9bt7BHFgyHvkXG47F/Qjj3sUJ6PNJkYYZqpTXSf+sAoCkpIoVSY7YtBhzH+ZuUmQhEsQKNITDUi7qXPFkGhwjmmEy4uvowwqVg/aYWfxUUVD2PpTFhUIFmryND3SzADhprFZShP09WYOE+R0049k8hIhM96cN9t2yyRcJWteJOUmNSp9EaGxqrUkf0uuUNh7sqHIbG0tfWsSbOnkUwVRNkLgitGZbXTXQ5sFX5C3zdN3Hwx6Jfp1cfPlx8uhnuLa/4hvAXayYyWn2tAqwp4IksFjJq7VDH47AKPEJL0KYqHYi3kMUaFsnoMi3PQVdGpppF53tx2hzbwcri6cbMYquXVrKqLY2J23owwVxoVW6biVatN+b/rGIDexCmmXwMSuhiPI6w7jMTWg9yTPAkUvM5lrrzaxIwGAo6JDhwJsvRKCKtYyMOJv7ZhfGTbsG5CdaZbQfbOLKvRxXnJs/aejikHxHoNz8c1k7zzbcl5LXQoyQbmeSYzGE5D3uZVzCU4bHcN9YFGEKe0AshLwKpij9cxtuW+nWMY+Zj+lrhPM1Qt+SDYZqt41FtZVme9bC2d+UAFb6BKhp3Vye6sGwW74Xvp7A8UiVVWlFwrHlHRRm7SGZgRCMrN2+okBrM8ynx28kZVWuUiuKhoD7IHfHEuyCPHgAF4KCURPzm0QpsUJD1Em7wHRZookRicp3r0kpJiPacPRmWHvXMqOzx5KycWH+dWSuoMRSdak27nhV36LpwGKBIaCKZBdN2vudbb2BxjlYFk54Y59y5fn9yOaEHb49m49vv2rmy7+6eCoNTCk//ev1Z1E+qno988rEOujtfXZ6J65Obm4m9c7PnxHs9f9LVysNnDqcj6EyhzXO9msZ2nNGqlEHrfLtcXtdfk0r0Sdj4vqUZyaE44pwbWfxHOMtpcgs8AHDWsv8HTXSQSIXCYyz4ZD73yR3DWGdDN7Uia6pXQA9OdyZ7tt2KhvxuHT6wCX9Q/LJnd1P2pbXnVnjNUkRuX9huy5YdtJSUTaTWSIr+a0nK0X9jAkUOpvnw2UYz/NYB7V1gjRqMPH5x0XlKIgt9eipBt0L7TEgqWachxx6bc8JOGFTc2vt2fGe9JaI3Qv1xzC273LEwgwY6RVGNTVI/RYLfQ0qOtkCmZ5i8KaljMvJ3pE9wBKx020nB3+GImrW6nc8B1E6K6APxzuzgoM2g5oohqR0LC6Acyqb6C9BH5PDiEnAKWZZUDXJ36/Ag1G99vTzVirZLg2JUyM0TTtaqVnhePx7FRPyw79Xorjej/eUrdlFBDctOvQFuWYL/Q4F81JuxXKTtp61B3HqCurUNGNTgZ/n8RlN9NVuSNYKrg6RD98pqAhcLGLr0G0hwT//6ZjTeoehZBV+9VMFCmq4nmyDx1QEji0jwXm6I+ujBAJmL0DLUn2AwGkR5Ch/TuYNnX4ycsG5vvcag/YCH80Ibrj12K9tZI1Pt1GIiJh8tc+gtMZ0ZX1XhozGrd4ZFAH6hvmqmOu4OQBjlIJjTFfE2jNKFewSp5y73OakJ14q34kIj2lnURG8pL/bed0rifi+/XwaThYdqEH8QtugHPQvk2XRJGgY0ewsCF8EHnwY8lA+AW+NbMNCgmHvENKXFmugRt1ZrSXndw0K+NUzz0ZZE675B6ZtEvMZzWulQx9SX0426KyEph1qQ7l0h7LkI2zx4EQIjwxA2Gbz8TVSjmrCchnXSSOceRDvjYADbLk5vC/VuFdfd1uudbnlYHaeyqtIGezXFy5ds1St3Fm1Vk909C7ALUjL4Nm3BT0rQu7O1uZVDHNgqoklP1I96a+2CTioaeiiwm+P3RVKbc7dq0+52bLV/sZ7eVnx212bNY7bnbogx2K3NOjH256/U78I0NtqwFlXTbYWNKTyiYl9T8VT/vSvabOlgnauzrZytXmdn2DqTwT2/uQZeRNsOjKzZBhzF1oI7R71kYZ1m2b9ue9DuZYkMmRGmW7y0TT9g8CMmd01ox/Z37cn0htWP1vtImFnmIN1N25uhIS+mjenzJPOt3msnEXWw26t8YRa1n3b/FxlUHeDeZe93HSBbYuuprRhW2z3jT/LOLAVv5vWq8W+FNEY7dgX+LalCQK4BjLWxIGPFX8RHIAHFz1zqzGLNmyOW/GzRSDPZ+4NZ6BddRz96h7OYiDyVtqA/ZspcvJN8Qbb2NfW4keQIJMbNfT9KQ/DvrZn47yj4gZ7iNv49Usj0mnIKsGUAjD7dFSPQ/+JEcXt8t3fl+t+ZqNfeO5xjCtY5TFjU8vl37USFy3oymTPNLcZ7Z1rPzezJ+I80TZ2rlSo7b+1MlGwVfFGraoVPZYwFu2MHHaqiWhpCCijkFPO4cKn9M+soTM9EfcSaCG4+X6NHeNMqhdHr0l+4ssm5om+DdYm8gV2T0Sq5pjFIHnbihBoBVd1/hkO2fAKzQ1E/H+MXVPpEkq0dQyLYb5DtdBUz3KNOGdUnGtrLN4cyuSPubMcsubGpXjfZJGrWH4NORM+5IYk6qTny1ljG6Pv+h+C6BOw2J0CWVVgHyDKWnAefSyN99KWJkw/+Bzrm61BTTAAA"
  }
}
# END EMBEDDED CONTEXT HELPERS
TOOLS = Path(tempfile.mkdtemp(prefix="ace_context_tools_"))
for name, payload in EMBEDDED_HELPERS.items():
    data = gzip.decompress(base64.b64decode(payload["gzip_base64"], validate=True))
    assert hashlib.sha256(data).hexdigest() == payload["sha256"], f"Helper checksum mismatch: {name}"
    (TOOLS / name).write_bytes(data)
# Explicitly load this notebook's helper, even if an earlier notebook imported one.
spec = importlib.util.spec_from_file_location("electric_full_song", TOOLS / "electric_full_song.py")
embedded_single = importlib.util.module_from_spec(spec)
spec.loader.exec_module(embedded_single)
sys.modules["electric_full_song"] = embedded_single
spec = importlib.util.spec_from_file_location("ace_context_embedded", TOOLS / "ace_context_chunks.py")
context = importlib.util.module_from_spec(spec)
spec.loader.exec_module(context)
ROOT = context.single.discover_root(Path.cwd(), globals().get("ROOT"))
if ROOT is None:
    raise RuntimeError("Checkout bridge belum ada di VM ini. Setup notebook 10 dahulu; notebook ini belum menyediakan model/GPU")

print("Upload satu file piano asli saja; tidak perlu ZIP tools")
uploaded = files.upload()
audios = [n for n in uploaded if Path(n).suffix.lower() in {".mp3", ".wav", ".flac", ".m4a", ".aac", ".ogg"}]
assert len(audios) == 1, "Pilih satu file piano asli"
SOURCE = TOOLS / ("piano" + Path(audios[0]).suffix.lower())
SOURCE.write_bytes(uploaded[audios[0]])
OUT = ROOT / "runs" / ("ace_context_" + uuid.uuid4().hex[:10])
plan = context.prepare(SOURCE, OUT, context=90, margin=6, fade=.58)
PLAN_PATH = OUT / "plan.json"
print("ROOT:", ROOT, "| PLAN:", PLAN_PATH)
for chunk in plan["chunks"]:
    print(chunk["index"], "input model:", chunk["model_duration"], "detik", "core:", round((chunk["core_end"]-chunk["core_start"])/32000, 3))
print("Tanpa SongMap terverifikasi: batas ini bukan downbeat atau bar yang diketahui")


## Satu probe durasi

Opsional: `90` sudah dipilih, sehingga tidak perlu diulang. `60` tersedia sebagai kontrol; `120` hanya untuk eksperimen eksplisit meskipun sudah dilaporkan berisik. Caption, seed, FP32, CFG dan langkah tetap sama. Dengarkan bagian waktu yang sama, jangan hanya melihat status generated. Jangan menjalankan beberapa probe sekaligus di T4 yang sama.


In [ ]:
from IPython.display import Audio, display

DURATION = 90  # 60 / 90 / 120
EXECUTE_PROBE = False  # True hanya pada runtime CUDA/bobot yang siap
PROBE_OUT = OUT / (f"probe_{DURATION}_" + uuid.uuid4().hex[:8])
probe_report = context.single.generate(ROOT, context.generation_source(OUT, plan), PROBE_OUT,
    start=0, duration=DURATION, execute=EXECUTE_PROBE)
print("Status:", probe_report["status"], "|", PROBE_OUT)
if EXECUTE_PROBE:
    display(Audio(filename=str(PROBE_OUT / "mix_preview.mp3"), normalize=False))


In [ ]:
# Jalankan setelah probe selesai, atau setelah gagal untuk report yang tersedia.
feedback = PROBE_OUT / "full_song_feedback.zip"
if feedback.is_file():
    files.download(str(feedback))
else:
    print("Belum ada feedback inferensi. Probe dry-run tidak membuat hasil model.")


## Chunk seluruh lagu, setelah durasi konteks dipilih

Plan awal membatasi **seluruh konteks model menjadi 90 detik termasuk margin**. Jika memilih cap lain, buat plan baru dengan `context.prepare(SOURCE, NEW_OUT, context=60, margin=6, fade=.58)` dan perbarui `PLAN_PATH`. Jangan mengganti plan yang telah memiliki hasil.

Generasi sequential dan memuat model lagi per chunk. Margin/reload menambah waktu. Chunk selesai dipakai kembali setelah verifikasi; proses diffusion yang gagal diulang sebagai attempt baru. Seed sama tidak menjamin timbre/register/groove sama.


In [ ]:
EXECUTE_CHUNKS = False
if EXECUTE_CHUNKS:
    progress = context.run_chunks(ROOT, PLAN_PATH)
    report = context.stitch_run(PLAN_PATH)
    print("Sambungan perlu didengar:", [s["index"] for s in report["seams"] if s["manual_review"]])
    display(Audio(filename=str(PLAN_PATH.parent / "stitched/mix_full.mp3"), normalize=False))
else:
    print("Tidak menjalankan inferensi atau stitching; plan tersimpan di", PLAN_PATH)


In [ ]:
# Paket kecil: mix/bass MP3, report seam dan cuplikan seam; tanpa bobot model.
stitched = PLAN_PATH.parent / "stitched"
if (stitched / "seams.json").is_file():
    bundle = PLAN_PATH.parent / "chunked_feedback.zip"
    with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.write(PLAN_PATH, "plan.json")
        for path in sorted(stitched.iterdir()):
            if path.suffix in {".json", ".mp3"} or path.name.startswith("seam_"):
                archive.write(path, "stitched/" + path.name)
    files.download(str(bundle))
else:
    print("Belum ada hasil stitching lengkap.")


Equal-power crossfade **tetap dapat membatalkan fase bass**. Helper mencari kandidat berdasarkan energi/korelasi, membuang tepi model, dan melaporkan perubahan level/pitch; ia tidak menjamin sambungan musikal. Birama/downbeat tidak ditebak dari BPM. Repaint dan conditioning BPM/key belum diterapkan. Lihat `ace-context-chunks.md` di tool ZIP untuk rincian.
